# Lesson 04 - Návrhový vzor Použitie nástrojov

V tejto lekcii sa naučíte návrhový vzor **Použitie nástrojov** pre AI agentov pomocou Microsoft Agent Framework (Python). Pokryjeme:

- Definovanie funkčných nástrojov pomocou dekorátora `@tool` a typizovaných parametrov
- Poskytovanie schém nástrojov, aby model vedel, čo každý nástroj robí
- Riadenie vykonávania nástrojov pomocou `approval_mode`
- Vracaní **štruktúrovaného výstupu** pomocou Pydantic modelov a `response_format`

Scenár je **agent na rezerváciu ciest**, ktorý môže vyhľadávať destinácie, kontrolovať dostupnosť a získavať informácie o letoch.


## Nastavenie


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Definovanie nástrojov pomocou dekorátora @tool

Dekorátor `@tool` premení obyčajnú Python funkciu na nástroj, ktorý môže agent zavolať.
Hlavné body:

- **Dokumentačný reťazec** sa stáva popisom nástroja, ktorý model vidí.
- **Typové anotácie** (vrátane `Annotated` s popismi) definujú schému nástroja.
- `approval_mode` kontroluje, či musí používateľ schváliť každé volanie pred jeho vykonaním.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Vytvorenie agenta s viacerými nástrojmi

Predajte všetky tri nástroje klientovi, aby model mohol vyvolať ktorékoľvek z nich podľa potreby na zodpovedanie otázky používateľa.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Štruktúrovaný výstup s nástrojmi

Nastavením `response_format` na model Pydantic je agent nútený vrátiť dobre typovaný JSON objekt namiesto voľného textu. Toto je užitočné, keď musí následný kód programovo spracovať výsledok.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Vzor schvaľovania nástrojov

Parameter `approval_mode` v `@tool` riadi, či volania nástroja vyžadujú schválenie človekom pred ich vykonaním:

| Režim | Správanie |
|---|---|
| `"never_require"` | Nástroj beží automaticky — nie je potrebné potvrdenie používateľa. |
| `"always_require"` | Každé volanie musí byť pred vykonaním schválené používateľom. |

Použite `"always_require"` pre nástroje, ktoré majú vedľajšie účinky (napr. rezervácia letu, zaúčtovanie kreditnej karty), aby človek zostal v procese.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Zhrnutie

V tejto lekcii ste sa naučili, ako:

1. **Definovať nástroje** pomocou dekorátora `@tool` s typovanými parametrami a dokumentačnými reťazcami, ktoré slúžia ako schéma nástroja.
2. **Zlúčiť viacero nástrojov** tak, aby ich agent mohol volať postupne na zodpovedanie zložitých dotazov.
3. **Vrátiť štruktúrovaný výstup** odovzdaním modelu Pydantic ako `response_format`.
4. **Ovládať schvaľovanie nástrojov** pomocou `approval_mode` na zabezpečenie ľudskej kontroly pri citlivých operáciách.

Tieto vzory tvoria základ pre tvorbu spoľahlivých agentov pripravených na produkciu, ktorí môžu bezpečne komunikovať s externými systémami.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Upozornenie**:  
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Aj keď sa snažíme o presnosť, prosím vezmite na vedomie, že automatizované preklady môžu obsahovať chyby alebo nepresnosti. Originálny dokument v jeho pôvodnom jazyku sa považuje za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za akékoľvek nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
